In [1]:
!pip -q install transformers==4.52.4 datasets==3.6.0 evaluate jiwer librosa soundfile accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 65.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is

In [2]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd

from datasets import Dataset, Audio

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [3]:
CSV_PATH = "/kaggle/input/datasets/organizations/mozillaorg/common-voice/cv-valid-dev.csv"

AUDIO_DIR = "/kaggle/input/datasets/organizations/mozillaorg/common-voice/cv-valid-dev/cv-valid-dev"

In [4]:
df = pd.read_csv(CSV_PATH)

print(df.shape)

df.head()

(4076, 8)


,filename,text,up_votes,down_votes,age,gender,accent,duration
0,cv-valid-dev/sample-000000.mp3,be careful with your prognostications said the...,1,0,NaN,NaN,NaN,NaN
1,cv-valid-dev/sample-000001.mp3,then why should they be surprised when they se...,2,0,NaN,NaN,NaN,NaN
2,cv-valid-dev/sample-000002.mp3,a young arab also loaded down with baggage ent...,2,0,NaN,NaN,NaN,NaN
3,cv-valid-dev/sample-000003.mp3,i thought that everything i owned would be des...,3,0,NaN,NaN,NaN,NaN
4,cv-valid-dev/sample-000004.mp3,he moved about invisible but everyone could he...,1,0,fourties,female,england,NaN


In [5]:
df = pd.read_csv(CSV_PATH)
print(df.shape)

(4076, 8)


In [6]:
df["filename"] = df["filename"].str.replace(
    "cv-valid-dev/",
    "",
    regex=False
)

df["audio"] = df["filename"].apply(
    lambda x: os.path.join(AUDIO_DIR, x)
)

In [7]:
missing = df[~df["audio"].apply(os.path.exists)]

print("Missing:", len(missing))

Missing: 0


In [8]:
df = df[["audio", "text"]]

df.head()

,audio,text
0,/kaggle/input/datasets/organizations/mozillaor...,be careful with your prognostications said the...
1,/kaggle/input/datasets/organizations/mozillaor...,then why should they be surprised when they se...
2,/kaggle/input/datasets/organizations/mozillaor...,a young arab also loaded down with baggage ent...
3,/kaggle/input/datasets/organizations/mozillaor...,i thought that everything i owned would be des...
4,/kaggle/input/datasets/organizations/mozillaor...,he moved about invisible but everyone could he...


In [9]:
dataset = Dataset.from_pandas(df)

In [10]:
dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

In [11]:
sample = dataset[0]

print(sample.keys())

print(sample["text"])

print(sample["audio"]["sampling_rate"])

print(len(sample["audio"]["array"]))

dict_keys(['audio', 'text'])
be careful with your prognostications said the stranger
16000
81024


In [12]:
print(dataset)

Dataset({
    features: ['audio', 'text'],
    num_rows: 4076
})


In [13]:
from transformers import (
    AutoProcessor,
    Wav2Vec2ForCTC,
)

2026-07-01 04:36:40.975795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782880601.243130      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782880601.316291      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782880601.894801      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782880601.894861      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782880601.894865      58 computation_placer.cc:177] computation placer alr

In [14]:
processor = AutoProcessor.from_pretrained(
    "facebook/wav2vec2-base-960h"
)

print(type(processor))
print(processor.tokenizer.vocab_size)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

<class 'transformers.models.wav2vec2.processing_wav2vec2.Wav2Vec2Processor'>
32


In [15]:
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base-960h",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
model.freeze_feature_encoder()

In [17]:
import re

def normalize(batch):
    text = batch["text"].upper()
    text = re.sub(r"[^A-Z' ]", "", text)
    batch["text"] = text
    return batch

dataset = dataset.map(normalize)

Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

In [18]:
def prepare_dataset(batch):

    audio = batch["audio"]

    inputs = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"]
    )

    batch["input_values"] = inputs.input_values[0]

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch

In [19]:
processed_dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    load_from_cache_file=False,
    desc="Preparing Dataset"
)

Preparing Dataset:   0%|          | 0/4076 [00:00<?, ? examples/s]

In [20]:
split = processed_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split["train"]
eval_dataset = split["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_values', 'labels'],
    num_rows: 3668
})
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 408
})


In [21]:
print(processor.tokenizer.vocab_size)

print(processed_dataset)

print(train_dataset)

print(eval_dataset)

print(train_dataset[0]["labels"][:20])

32
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 4076
})
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 3668
})
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 408
})
[10, 6, 4, 15, 8, 8, 26, 5, 14, 4, 15, 10, 26, 5, 4, 7, 4, 13, 16, 12]


In [22]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features):

        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch

In [23]:
data_collator = DataCollatorCTCWithPadding(
    processor=processor
)

In [24]:
batch = data_collator([
    train_dataset[0],
    train_dataset[1]
])

print(batch.keys())

print(batch["input_values"].shape)

print(batch["labels"].shape)

dict_keys(['input_values', 'labels'])
torch.Size([2, 59904])
torch.Size([2, 32])


In [25]:
!pip -q install evaluate jiwer

In [26]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")

In [28]:
def compute_metrics(pred):

    pred_ids = np.argmax(pred.predictions, axis=-1)

    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"wer": wer}

In [29]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./wav2vec2_commonvoice",

    # Train & Evaluation
    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    # Learning
    learning_rate=1e-5,
    warmup_steps=100,

    # Batch Size
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=2,

    # Epochs
    num_train_epochs=10,

    # Logging
    logging_strategy="steps",
    logging_steps=50,

    # IMPORTANT
    fp16=False,

    # Save Best Model
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Other
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [31]:
print(len(train_dataset))
print(len(eval_dataset))

3668
408


In [32]:
print(training_args.fp16)
print(training_args.learning_rate)
print(training_args.per_device_train_batch_size)

False
1e-05
2


In [33]:
print(len(dataset))
print(len(processed_dataset))
print(len(train_dataset))
print(len(eval_dataset))

4076
4076
3668
408


In [34]:
bad = 0

for i in range(len(processed_dataset)):
    if (
        len(processed_dataset[i]["input_values"]) == 0 or
        len(processed_dataset[i]["labels"]) == 0
    ):
        bad += 1
        print("Bad sample:", i)

print("Total bad samples:", bad)

Total bad samples: 0


In [35]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print(df["text"].isna().sum())
print(df["text"].eq("").sum())

0
0


In [36]:
processed_dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    load_from_cache_file=False,
)

Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

In [37]:
trainer.train()

Epoch,Training Loss,Validation Loss,Wer
1,0.452700,0.217269,0.117521
2,0.341400,0.213304,0.107639
3,0.343300,0.202534,0.103365
4,0.280600,0.193932,0.094818
5,0.316000,0.189578,0.100160
6,0.278200,0.192585,0.095620
7,0.294900,0.190001,0.093750
8,0.323900,0.184542,0.094017
9,0.286800,0.186381,0.093750
10,0.272700,0.185120,0.093483


TrainOutput(global_step=4590, training_loss=0.3438577531469673, metrics={'train_runtime': 3902.917, 'train_samples_per_second': 9.398, 'train_steps_per_second': 1.176, 'total_flos': 2.170208237040599e+18, 'train_loss': 0.3438577531469673, 'epoch': 10.0})

In [40]:
trainer.save_model(
    "/kaggle/working/wav2vec2_commonvoice"
)

processor.save_pretrained(
    "/kaggle/working/wav2vec2_commonvoice"
)

[]